# Flood Mapping — Corrected Kuro Siwo Training and Evaluation

This is a **standalone, editable training/evaluation notebook** reconstructed from the training portion of the provided PDF export. It applies the main fixes discussed:

- Keep the six-channel, three-class U-Net contract and current SAR normalization.
- Fail visibly if ImageNet pretrained weights cannot be loaded; no silent random-initialization fallback.
- Choose the best checkpoint using **fixed argmax flood IoU**, not a newly optimized threshold every epoch.
- Use validation-driven `ReduceLROnPlateau`.
- Calibrate the flood-probability threshold **once after checkpoint selection** on validation data.
- Evaluate the held-out test split at that fixed validation threshold, and report argmax metrics separately.
- Keep atomic, resumable training checkpoints and deterministic mid-epoch replay.
- Count class pixels from training shards only before optionally enabling class weighting.

**Scope note:** the only source uploaded was a 51-page PDF, not the original `.ipynb`. This notebook contains the corrected model-training/evaluation workflow; it does **not** recreate every satellite acquisition, GIS overlay, OSM routing, dashboard, and EMSR927 cell from the full end-to-end notebook. Upload the original `.ipynb` to get the entire pipeline patched without risking dropped or corrupted geospatial code.

## Before running

1. Confirm `PROJECT_ROOT` points to the folder containing `dataset/train_grd`, `dataset/val_grd`, and `dataset/test_grd`.
2. Install a PyTorch/torchvision pair compatible with your CUDA setup separately; this notebook does not reinstall PyTorch.
3. Run cells in order. Since checkpoint selection/scheduling changed, set `RESET_TRAINING = True` for the first run of this version, then set it back to `False` before any resume.
4. The saved model filename remains `best_unet_resnet50.pth` for compatibility with the downstream inference cell.


In [1]:
# Install geospatial/model packages only if missing. Do not upgrade torch/torchvision here.
%pip install -q segmentation-models-pytorch scipy scikit-learn matplotlib numpy pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import io
import json
import math
import time
import random
import tarfile
import hashlib
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader
import segmentation_models_pytorch as smp

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------- Local paths ----------------
PROJECT_ROOT = Path(os.environ.get(
    "FLOOD_PROJECT_ROOT",
    r"C:\NIT KKR\after placement\iit mandi hackathon"
))
DATA_DIR = PROJECT_ROOT / "dataset"
TRAIN_DIR = DATA_DIR / "train_grd"
VAL_DIR = DATA_DIR / "val_grd"
TEST_DIR = DATA_DIR / "test_grd"
OUT_DIR = PROJECT_ROOT / "outputs_e2e"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = OUT_DIR / "best_unet_resnet50.pth"
LATEST_CKPT_PATH = OUT_DIR / "latest_training_checkpoint.pt"
META_PATH = OUT_DIR / "model_metadata.json"
HISTORY_PATH = OUT_DIR / "training_history.json"
TEST_REPORT_PATH = OUT_DIR / "kuro_siwo_test_report.json"

# ---------------- Dataset/model contract ----------------
CHANNELS = ["sec1_vv", "sec1_vh", "sec2_vv", "sec2_vh", "flood_vv", "flood_vh"]
MASK_KEY = "mask"
VALID_KEY = "valid_mask"
CLASS_NAMES = ["no_water", "permanent_water", "flood"]
NUM_CLASSES = 3

# Official Kuro Siwo GRD scaling used in the supplied PDF. Keep in this order.
SAR_MEAN = np.array([0.0953, 0.0264] * 3, dtype=np.float32)
SAR_STD = np.array([0.0427, 0.0215] * 3, dtype=np.float32)
CLIP_MAX = 0.15

# ---------------- Training settings ----------------
SEED = 42
EPOCHS = 20
BATCH_SIZE = 4
LR = 2e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0  # Keep 0 for deterministic mid-epoch resume.
SHUFFLE_BUFFER = 64
CHECKPOINT_EVERY_STEPS = 10
PATIENCE = 5
ENCODER_NAME = "resnet50"  # Run resnet18 later as a controlled ablation.
IMAGENET_PRETRAINED = True
USE_CLASS_WEIGHTS = False
CLASS_WEIGHTS = None  # Filled from training-only counts if enabled.
RESET_TRAINING = False  # Set True ONCE for this changed scheduler/selection policy.
EVAL_TTA_ON_TEST = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset directory exists: {DATA_DIR.exists()}")
print(f"Output directory: {OUT_DIR}")
print("Model contract: 6 linear-power SAR channels -> 3 classes; DEM is not a model input.")

Device: cuda
Project root: C:\NIT KKR\after placement\iit mandi hackathon
Dataset directory exists: True
Output directory: C:\NIT KKR\after placement\iit mandi hackathon\outputs_e2e
Model contract: 6 linear-power SAR channels -> 3 classes; DEM is not a model input.


In [3]:
def discover_shards(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    found = []
    for pattern in ("*.tar", "*.tar.gz", "*.tgz"):
        found.extend(folder.glob(pattern))
    return sorted(set(found))


def iter_shard(path):
    """Yield (sample_id, {array_key: np.ndarray}) without extracting archive files."""
    with tarfile.open(str(path), "r:*") as tf:
        groups = {}
        for member in tf.getmembers():
            if not member.isfile():
                continue
            base = Path(member.name).name
            if not base.endswith(".npy") or "." not in base:
                continue
            sample_id, key_file = base.split(".", 1)
            key = key_file[:-4]
            groups.setdefault(sample_id, {})[key] = member

        for sample_id in sorted(groups):
            sample = {}
            for key, member in groups[sample_id].items():
                stream = tf.extractfile(member)
                if stream is None:
                    continue
                with stream:
                    sample[key] = np.load(io.BytesIO(stream.read()), allow_pickle=False)
            if sample:
                yield sample_id, sample


def unpack_sample(sample):
    missing = [key for key in CHANNELS + [MASK_KEY] if key not in sample]
    if missing:
        raise KeyError(f"Missing keys {missing}; found keys: {sorted(sample)}")

    x = np.stack([np.squeeze(sample[key]).astype(np.float32) for key in CHANNELS], axis=0)
    y_raw = np.squeeze(sample[MASK_KEY])
    finite_mask = np.isfinite(y_raw)
    y = np.where(finite_mask, y_raw, -1).astype(np.int64)

    if VALID_KEY in sample:
        valid_flag = np.squeeze(sample[VALID_KEY]).astype(bool)
    else:
        valid_flag = np.ones(y.shape, dtype=bool)

    valid = valid_flag & finite_mask & (y >= 0) & (y < NUM_CLASSES) & np.isfinite(x).all(axis=0)
    return x, y, valid


def normalize_sar_numpy(x):
    """Same clipping/normalization used for both training and scene inference."""
    x = np.nan_to_num(np.asarray(x, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    x = np.clip(x, 0.0, CLIP_MAX)
    x = (x - SAR_MEAN[:, None, None]) / SAR_STD[:, None, None]
    return np.clip(np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0), -8.0, 8.0).astype(np.float32)


def preprocess_sample(sample):
    x, y, valid = unpack_sample(sample)
    x_norm = normalize_sar_numpy(x)
    target = y.copy()
    target[~valid] = 255  # CrossEntropyLoss ignore_index
    return torch.from_numpy(x_norm), torch.from_numpy(target), torch.from_numpy(valid.astype(np.bool_))


class KuroDataset(IterableDataset):
    """Iterable tar-shard reader with deterministic ordering for safe resume."""
    def __init__(self, shards, shuffle=False, buffer=SHUFFLE_BUFFER, seed=SEED):
        super().__init__()
        self.shards = [str(path) for path in shards]
        self.shuffle = bool(shuffle)
        self.buffer = max(1, int(buffer))
        self.seed = int(seed)
        self.epoch = 1

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)
        shards = list(self.shards)
        if self.shuffle:
            rng.shuffle(shards)
        buffer = []
        for shard in shards:
            for _, sample in iter_shard(shard):
                item = preprocess_sample(sample)
                if not self.shuffle:
                    yield item
                    continue
                buffer.append(item)
                if len(buffer) >= self.buffer:
                    idx = rng.randrange(len(buffer))
                    buffer[idx], buffer[-1] = buffer[-1], buffer[idx]
                    yield buffer.pop()
        if self.shuffle:
            rng.shuffle(buffer)
            while buffer:
                yield buffer.pop()


train_shards = discover_shards(TRAIN_DIR)
val_shards = discover_shards(VAL_DIR)
test_shards = discover_shards(TEST_DIR)
if not train_shards:
    raise FileNotFoundError(f"No training .tar shards found in {TRAIN_DIR}")
if not test_shards:
    raise FileNotFoundError(f"No held-out test .tar shards found in {TEST_DIR}")
if not val_shards:
    if len(train_shards) < 3:
        raise FileNotFoundError(f"No validation shards in {VAL_DIR}; need at least 3 training shards to hold some out.")
    split_shards = train_shards.copy()
    random.Random(SEED).shuffle(split_shards)
    n_val = max(1, int(round(0.15 * len(split_shards))))
    val_shards, train_shards = split_shards[:n_val], split_shards[n_val:]
    print(f"[INFO] VAL_DIR has no shards; held out {len(val_shards)} shard(s) for validation.")

if len(val_shards) < 2:
    warnings.warn("Validation contains fewer than two shards. Validation-based checkpoint selection may be noisy.")

print(f"Train shards: {len(train_shards)} | validation shards: {len(val_shards)} | test shards: {len(test_shards)}")
first_id, first_sample = next(iter_shard(train_shards[0]))
x_check, y_check, valid_check = unpack_sample(first_sample)
print(f"Sample {first_id}: X={x_check.shape}, mask={y_check.shape}, valid fraction={valid_check.mean():.3f}")
print("Valid labels in first sample:", np.unique(y_check[valid_check]).tolist())
if x_check.shape[0] != len(CHANNELS):
    raise ValueError(f"Expected {len(CHANNELS)} input channels, got {x_check.shape[0]}")
if not np.isin(np.unique(y_check[valid_check]), np.arange(NUM_CLASSES)).all():
    raise ValueError("Unexpected class ID found in valid labels.")

train_dataset = KuroDataset(train_shards, shuffle=True, buffer=SHUFFLE_BUFFER, seed=SEED)
val_dataset = KuroDataset(val_shards, shuffle=False, seed=SEED)
test_dataset = KuroDataset(test_shards, shuffle=False, seed=SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))

Train shards: 4 | validation shards: 1 | test shards: 3


C:\Users\Vidhy\AppData\Local\Temp\ipykernel_24052\2291894191.py:125: UserWarning: Validation contains fewer than two shards. Validation-based checkpoint selection may be noisy.
  warnings.warn("Validation contains fewer than two shards. Validation-based checkpoint selection may be noisy.")


Sample 000000: X=(6, 224, 224), mask=(224, 224), valid fraction=1.000
Valid labels in first sample: [0, 2]


In [4]:
def count_training_pixels(shards):
    """Count valid labels on TRAIN shards only. Never use validation/test labels for weights."""
    counts = np.zeros(NUM_CLASSES, dtype=np.int64)
    for shard in shards:
        for _, sample in iter_shard(shard):
            _, y, valid = unpack_sample(sample)
            labels = y[valid]
            if labels.size:
                counts += np.bincount(labels, minlength=NUM_CLASSES)[:NUM_CLASSES]
    if counts.sum() == 0:
        raise RuntimeError("No valid labelled pixels were found in the training shards.")
    return counts


def inverse_sqrt_class_weights(counts, max_weight=3.0):
    counts = np.asarray(counts, dtype=np.float64)
    if counts.shape != (NUM_CLASSES,) or np.any(counts <= 0):
        raise ValueError("Every class must have positive training-pixel count to compute weights.")
    frequencies = counts / counts.sum()
    weights = 1.0 / np.sqrt(frequencies)
    weights /= weights.mean()
    weights = np.clip(weights, 0.5, max_weight)
    weights /= weights.mean()
    return weights.astype(np.float32)

train_class_counts = count_training_pixels(train_shards)
print("Training pixels by class:", dict(zip(CLASS_NAMES, train_class_counts.tolist())))
print("Training class fractions:", (train_class_counts / train_class_counts.sum()).round(6).tolist())
if np.any(train_class_counts == 0):
    absent = [CLASS_NAMES[i] for i, n in enumerate(train_class_counts) if n == 0]
    raise RuntimeError(f"Training split has no labelled pixels for class(es): {absent}")

if USE_CLASS_WEIGHTS:
    CLASS_WEIGHTS = inverse_sqrt_class_weights(train_class_counts).tolist()
    print("Training-derived weights:", dict(zip(CLASS_NAMES, np.round(CLASS_WEIGHTS, 3).tolist())))
else:
    CLASS_WEIGHTS = None
    print("Class weighting disabled for baseline. Run a separate weighted experiment only after recording this baseline.")

Training pixels by class: {'no_water': 929196282, 'permanent_water': 62471701, 'flood': 199031390}
Training class fractions: [0.780379, 0.052466, 0.167155]
Class weighting disabled for baseline. Run a separate weighted experiment only after recording this baseline.


In [5]:
def build_model(pretrained=True):
    """Fail explicitly if requested pretrained weights cannot be obtained."""
    encoder_weights = "imagenet" if pretrained else None
    model = smp.Unet(
        encoder_name=ENCODER_NAME,
        encoder_weights=encoder_weights,
        in_channels=len(CHANNELS),
        classes=NUM_CLASSES,
        activation=None,
    )
    return model.to(DEVICE)


def metrics_from_counts(tp, fp, fn):
    tp, fp, fn = float(tp), float(fp), float(fn)
    return {
        "iou": tp / max(tp + fp + fn, 1.0),
        "f1": (2.0 * tp) / max(2.0 * tp + fp + fn, 1.0),
        "precision": tp / max(tp + fp, 1.0),
        "recall": tp / max(tp + fn, 1.0),
        "tp": int(tp), "fp": int(fp), "fn": int(fn),
    }


def class_metrics_from_confusion(confusion):
    confusion = np.asarray(confusion, dtype=np.int64)
    result = {}
    for cls, name in enumerate(CLASS_NAMES):
        tp = int(confusion[cls, cls])
        fp = int(confusion[:, cls].sum() - tp)
        fn = int(confusion[cls, :].sum() - tp)
        result[name] = metrics_from_counts(tp, fp, fn)
    result["mean_iou"] = float(np.mean([result[name]["iou"] for name in CLASS_NAMES]))
    result["confusion_matrix_rows_true_cols_pred"] = confusion.tolist()
    return result


@torch.no_grad()
def predict_probs(model, x, tta=False):
    if not tta:
        return F.softmax(model(x), dim=1)
    predictions = []
    for dims in (None, (-1,), (-2,), (-1, -2)):
        x_aug = x if dims is None else x.flip(dims)
        probs = F.softmax(model(x_aug), dim=1)
        if dims is not None:
            probs = probs.flip(dims)
        predictions.append(probs)
    return torch.stack(predictions, dim=0).mean(dim=0)


@torch.no_grad()
def evaluate_model(model, loader, thresholds=None, tta=False):
    """Report multiclass argmax metrics plus binary flood-probability threshold metrics."""
    if thresholds is None:
        thresholds = np.round(np.arange(0.10, 0.91, 0.05), 2)
    thresholds = [float(t) for t in thresholds]
    model.eval()
    confusion = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.int64, device=DEVICE)
    tp = torch.zeros(len(thresholds), dtype=torch.float64, device=DEVICE)
    fp = torch.zeros_like(tp)
    fn = torch.zeros_like(tp)
    valid_total = 0

    for x, y, valid in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        valid = valid.to(DEVICE, non_blocking=True) & (y != 255) & (y >= 0) & (y < NUM_CLASSES)
        probs = predict_probs(model, x, tta=tta)
        pred = probs.argmax(dim=1)
        if valid.any():
            hist = torch.bincount(y[valid] * NUM_CLASSES + pred[valid], minlength=NUM_CLASSES**2)
            confusion += hist.reshape(NUM_CLASSES, NUM_CLASSES)
            p_flood = probs[:, 2][valid]
            y_flood = y[valid] == 2
            valid_total += int(valid.sum().item())
            for idx, threshold in enumerate(thresholds):
                positive = p_flood >= threshold
                tp[idx] += (positive & y_flood).sum().double()
                fp[idx] += (positive & ~y_flood).sum().double()
                fn[idx] += (~positive & y_flood).sum().double()

    if valid_total == 0:
        raise RuntimeError("Evaluation has no valid labelled pixels.")

    argmax_metrics = class_metrics_from_confusion(confusion.detach().cpu().numpy())
    threshold_scores = []
    for idx, threshold in enumerate(thresholds):
        threshold_scores.append({"threshold": threshold, **metrics_from_counts(tp[idx].item(), fp[idx].item(), fn[idx].item())})
    best_threshold = max(threshold_scores, key=lambda row: (row["iou"], row["f1"]))
    return {
        "argmax": argmax_metrics,
        "threshold_sweep": threshold_scores,
        "best_threshold": best_threshold,
        "valid_pixels": valid_total,
        "tta": bool(tta),
    }


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as handle:
        json.dump(obj, handle, indent=2, default=lambda value: value.item() if hasattr(value, "item") else str(value))
    os.replace(tmp, path)


def load_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return {} if default is None else default
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def atomic_torch_save(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def capture_rng_state():
    state = {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state()}
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state


def restore_rng_state(state):
    if not state:
        return
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and state.get("cuda"):
        try:
            torch.cuda.set_rng_state_all(state["cuda"])
        except Exception as exc:
            warnings.warn(f"CUDA RNG state could not be restored exactly: {exc}")


def get_config_signature():
    payload = {
        "model": f"smp.Unet-{ENCODER_NAME}-6ch-3class-v2-fixed-selection",
        "channels": CHANNELS,
        "sar_mean": SAR_MEAN.tolist(), "sar_std": SAR_STD.tolist(), "clip_max": CLIP_MAX,
        "batch_size": BATCH_SIZE, "seed": SEED, "epochs": EPOCHS, "lr": LR,
        "weight_decay": WEIGHT_DECAY, "shuffle_buffer": SHUFFLE_BUFFER,
        "class_weights_enabled": USE_CLASS_WEIGHTS, "class_weights": CLASS_WEIGHTS,
        "train_shards": [(Path(path).name, Path(path).stat().st_size) for path in train_shards],
        "val_shards": [(Path(path).name, Path(path).stat().st_size) for path in val_shards],
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()

print("Model/metrics/checkpoint helpers ready.")

Model/metrics/checkpoint helpers ready.


In [6]:
# ---------------- Initialize the run ----------------
if RESET_TRAINING:
    for path in (LATEST_CKPT_PATH, BEST_MODEL_PATH, META_PATH, HISTORY_PATH):
        if path.exists():
            path.unlink()
    print("[RESET] Removed old training checkpoint/model metadata/history. Set RESET_TRAINING=False before resuming.")

resume_exists = LATEST_CKPT_PATH.exists() and not RESET_TRAINING
model = build_model(pretrained=(IMAGENET_PRETRAINED and not resume_exists))
loss_weight = torch.tensor(CLASS_WEIGHTS, dtype=torch.float32, device=DEVICE) if USE_CLASS_WEIGHTS else None
criterion = nn.CrossEntropyLoss(weight=loss_weight, ignore_index=255)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2,
    threshold=1e-3, threshold_mode="abs", min_lr=1e-6,
)
use_amp = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

history = []
best_val_iou = -1.0
epochs_without_improvement = 0
global_step = 0
start_epoch = 1
resume_batch_idx = 0
resume_running_loss_sum = 0.0
resume_running_steps = 0
early_stop_reached = False
config_signature = get_config_signature()

if resume_exists:
    print(f"[RESUME] Loading {LATEST_CKPT_PATH}")
    checkpoint = torch.load(LATEST_CKPT_PATH, map_location=DEVICE, weights_only=False)
    if checkpoint.get("config_signature") != config_signature:
        raise RuntimeError(
            "Checkpoint configuration differs from this notebook. For the first run of the corrected policy, "
            "set RESET_TRAINING=True once, run this cell, then set it back to False before resuming."
        )
    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    if checkpoint.get("scaler_state"):
        scaler.load_state_dict(checkpoint["scaler_state"])
    history = checkpoint.get("history", [])
    best_val_iou = float(checkpoint.get("best_val_iou", -1.0))
    epochs_without_improvement = int(checkpoint.get("epochs_without_improvement", 0))
    global_step = int(checkpoint.get("global_step", 0))
    start_epoch = int(checkpoint.get("epoch", 1))
    resume_batch_idx = int(checkpoint.get("next_batch_idx", 0))
    resume_running_loss_sum = float(checkpoint.get("running_loss_sum", 0.0))
    resume_running_steps = int(checkpoint.get("running_steps", 0))
    early_stop_reached = bool(checkpoint.get("early_stop_reached", False))
    restore_rng_state(checkpoint.get("rng_state"))
    print(f"[RESUME] epoch={start_epoch}, next batch={resume_batch_idx}, global step={global_step}, best fixed IoU={best_val_iou:.4f}")
else:
    print("[START] No resumable checkpoint found; starting new training run.")


def create_checkpoint(epoch, next_batch_idx, running_loss_sum, running_steps):
    return {
        "format_version": 2,
        "config_signature": config_signature,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
        "epoch": int(epoch), "next_batch_idx": int(next_batch_idx),
        "running_loss_sum": float(running_loss_sum), "running_steps": int(running_steps),
        "global_step": int(global_step), "best_val_iou": float(best_val_iou),
        "epochs_without_improvement": int(epochs_without_improvement),
        "early_stop_reached": bool(early_stop_reached), "history": history,
        "rng_state": capture_rng_state(), "saved_at": datetime.now().isoformat(timespec="seconds"),
    }

[START] No resumable checkpoint found; starting new training run.


In [7]:
# ---------------- Train with fixed-metric checkpoint selection ----------------
if start_epoch <= EPOCHS and not early_stop_reached:
    for epoch in range(start_epoch, EPOCHS + 1):
        train_dataset.set_epoch(epoch)
        model.train()
        batch_offset = resume_batch_idx if epoch == start_epoch else 0
        loss_sum = resume_running_loss_sum if epoch == start_epoch else 0.0
        loss_steps = resume_running_steps if epoch == start_epoch else 0
        if batch_offset:
            print(f"[RESUME] Replaying deterministic epoch {epoch} order and skipping {batch_offset} completed batch(es).")
        epoch_t0 = time.time()

        for batch_idx, (x, y, valid) in enumerate(train_loader):
            if batch_idx < batch_offset:
                continue
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            # Deterministic spatial flips: transform all six dates and label identically.
            aug_rng = random.Random(SEED + epoch * 1_000_003 + batch_idx * 9_176)
            if aug_rng.random() < 0.5:
                x, y = x.flip(-1), y.flip(-1)
            if aug_rng.random() < 0.5:
                x, y = x.flip(-2), y.flip(-2)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                logits = model(x)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()

            loss_value = float(loss.detach().cpu())
            loss_sum += loss_value
            loss_steps += 1
            global_step += 1

            if batch_idx == 0 or (batch_idx + 1) % 25 == 0:
                print(f"Epoch {epoch:02d}/{EPOCHS:02d} | batch {batch_idx + 1} | loss={loss_value:.4f} | step={global_step}")

            if CHECKPOINT_EVERY_STEPS > 0 and global_step % CHECKPOINT_EVERY_STEPS == 0:
                atomic_torch_save(create_checkpoint(epoch, batch_idx + 1, loss_sum, loss_steps), LATEST_CKPT_PATH)

        # Validate before scheduler/checkpoint decision. The metric used for selection never changes definition.
        val_metrics = evaluate_model(model, val_loader, tta=False)
        val_argmax_iou = float(val_metrics["argmax"]["flood"]["iou"])
        val_best_threshold_diagnostic = val_metrics["best_threshold"]  # diagnostic only; NOT used to select epoch
        scheduler.step(val_argmax_iou)
        epoch_loss = loss_sum / max(loss_steps, 1)
        improved = val_argmax_iou > best_val_iou + 1e-4

        record = {
            "epoch": epoch,
            "train_loss": epoch_loss,
            "val_flood_iou_argmax": val_argmax_iou,
            "val_flood_f1_argmax": float(val_metrics["argmax"]["flood"]["f1"]),
            "val_threshold_diagnostic": float(val_best_threshold_diagnostic["threshold"]),
            "val_threshold_iou_diagnostic": float(val_best_threshold_diagnostic["iou"]),
            "val_threshold_f1_diagnostic": float(val_best_threshold_diagnostic["f1"]),
            "lr": float(optimizer.param_groups[0]["lr"]),
            "epoch_seconds": float(time.time() - epoch_t0),
        }
        history.append(record)
        print(
            f"Epoch {epoch:02d}/{EPOCHS:02d} | loss={epoch_loss:.4f} | "
            f"val flood IoU(argmax)={val_argmax_iou:.4f} | "
            f"F1(argmax)={val_metrics['argmax']['flood']['f1']:.4f} | "
            f"diagnostic best threshold={val_best_threshold_diagnostic['threshold']:.2f} | "
            f"LR={optimizer.param_groups[0]['lr']:.2e}"
        )

        if improved:
            best_val_iou = val_argmax_iou
            epochs_without_improvement = 0
            atomic_torch_save(model.state_dict(), BEST_MODEL_PATH)
            save_json({
                "model": f"U-Net {ENCODER_NAME}, six temporal SAR channels, three classes",
                "encoder_name": ENCODER_NAME,
                "channels": CHANNELS,
                "class_names": CLASS_NAMES,
                "best_epoch": epoch,
                "best_validation_flood_iou_argmax": best_val_iou,
                "threshold_selection_policy": "Calibrated once after checkpoint selection on validation; not used for epoch selection.",
                "normalization": {"mean": SAR_MEAN.tolist(), "std": SAR_STD.tolist(), "clip_max": CLIP_MAX},
                "training_config": {"epochs_requested": EPOCHS, "batch_size": BATCH_SIZE, "lr": LR,
                                     "weight_decay": WEIGHT_DECAY, "use_class_weights": USE_CLASS_WEIGHTS,
                                     "class_weights": CLASS_WEIGHTS if USE_CLASS_WEIGHTS else None},
                "warning": "Validation is small; transfer to Himalayan terrain remains uncertain.",
            }, META_PATH)
        else:
            epochs_without_improvement += 1

        save_json(history, HISTORY_PATH)
        resume_batch_idx, resume_running_loss_sum, resume_running_steps = 0, 0.0, 0
        early_stop_reached = epochs_without_improvement >= PATIENCE
        atomic_torch_save(create_checkpoint(epoch + 1, 0, 0.0, 0), LATEST_CKPT_PATH)
        if early_stop_reached:
            print(f"[EARLY STOP] No improvement in fixed argmax flood IoU for {PATIENCE} epochs. Best={best_val_iou:.4f}")
            break
elif early_stop_reached:
    print("[RESUME] This run already reached early stopping. Set RESET_TRAINING=True only to intentionally restart.")
else:
    print(f"[RESUME] Checkpoint is already at epoch {start_epoch}; requested EPOCHS={EPOCHS}. Nothing to train.")

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError("No best model exists. Check the dataset and training logs.")
print(f"Best model: {BEST_MODEL_PATH}")
print(f"Latest resume state: {LATEST_CKPT_PATH}")
print(f"Training history: {HISTORY_PATH}")

Epoch 01/20 | batch 1 | loss=1.5318 | step=1
Epoch 01/20 | batch 25 | loss=0.7756 | step=25
Epoch 01/20 | batch 50 | loss=0.6769 | step=50
Epoch 01/20 | batch 75 | loss=0.4642 | step=75
Epoch 01/20 | batch 100 | loss=0.7106 | step=100
Epoch 01/20 | batch 125 | loss=0.3728 | step=125
Epoch 01/20 | batch 150 | loss=0.3570 | step=150
Epoch 01/20 | batch 175 | loss=0.3305 | step=175
Epoch 01/20 | batch 200 | loss=0.2730 | step=200
Epoch 01/20 | batch 225 | loss=0.4054 | step=225
Epoch 01/20 | batch 250 | loss=0.3028 | step=250
Epoch 01/20 | batch 275 | loss=0.8483 | step=275
Epoch 01/20 | batch 300 | loss=0.3478 | step=300
Epoch 01/20 | batch 325 | loss=0.2809 | step=325
Epoch 01/20 | batch 350 | loss=0.3144 | step=350
Epoch 01/20 | batch 375 | loss=0.4310 | step=375
Epoch 01/20 | batch 400 | loss=0.2928 | step=400
Epoch 01/20 | batch 425 | loss=0.3367 | step=425
Epoch 01/20 | batch 450 | loss=0.2090 | step=450
Epoch 01/20 | batch 475 | loss=0.2490 | step=475
Epoch 01/20 | batch 500 | loss

In [8]:
# ---------------- Calibrate threshold ONCE on validation using best checkpoint ----------------
best_state = torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(best_state)
model.eval()
val_calibration = evaluate_model(model, val_loader, tta=False)
selected_threshold_row = val_calibration["best_threshold"]
selected_threshold = float(selected_threshold_row["threshold"])

meta = load_json(META_PATH, default={})
meta["best_threshold"] = selected_threshold
meta["threshold_selection_policy"] = "Selected once on validation using the fixed-metric best checkpoint; never used for checkpoint selection."
meta["threshold_validation_metrics"] = selected_threshold_row
save_json(meta, META_PATH)

print(
    f"Validation-calibrated flood threshold={selected_threshold:.2f} | "
    f"IoU={selected_threshold_row['iou']:.4f} | F1={selected_threshold_row['f1']:.4f}"
)
print("Important: do not re-select this threshold on the held-out test split or EMSR927.")

Validation-calibrated flood threshold=0.30 | IoU=0.7774 | F1=0.8747
Important: do not re-select this threshold on the held-out test split or EMSR927.


In [9]:
# ---------------- Held-out test evaluation: report, do not tune ----------------
# Test metrics are computed at the validation-selected threshold only. No test threshold sweep is used for selection.
test_metrics = evaluate_model(model, test_loader, thresholds=[selected_threshold], tta=EVAL_TTA_ON_TEST)
test_argmax_flood = test_metrics["argmax"]["flood"]
test_threshold_flood = test_metrics["threshold_sweep"][0]

test_report = {
    "model": f"U-Net {ENCODER_NAME}, six temporal SAR channels, three classes",
    "test_argmax_metrics": test_metrics["argmax"],
    "test_metrics_at_validation_threshold": test_threshold_flood,
    "validation_selected_threshold": selected_threshold,
    "tta": bool(test_metrics["tta"]),
    "test_shards": [Path(path).name for path in test_shards],
    "warning": "Held-out Kuro Siwo results do not prove generalization to the Trishuli/Himalayan scene.",
}
save_json(test_report, TEST_REPORT_PATH)

# Stable schema for downstream results.json consumers: primary figures match threshold_used;
# argmax figures are preserved under explicit names.
results_path = OUT_DIR / "results.json"
results = load_json(results_path, default={})
results["kuro_siwo_test"] = {
    "iou": float(test_threshold_flood["iou"]),
    "f1": float(test_threshold_flood["f1"]),
    "precision": float(test_threshold_flood["precision"]),
    "recall": float(test_threshold_flood["recall"]),
    "threshold_used": selected_threshold,
    "argmax_iou": float(test_argmax_flood["iou"]),
    "argmax_f1": float(test_argmax_flood["f1"]),
    "argmax_precision": float(test_argmax_flood["precision"]),
    "argmax_recall": float(test_argmax_flood["recall"]),
    "tta": bool(test_metrics["tta"]),
}
save_json(results, results_path)

print(json.dumps({
    "flood_argmax": test_argmax_flood,
    "flood_at_validation_threshold": test_threshold_flood,
    "report_path": str(TEST_REPORT_PATH),
}, indent=2))

{
  "flood_argmax": {
    "iou": 0.5306669627732583,
    "f1": 0.6933800437056502,
    "precision": 0.6808611689239478,
    "recall": 0.7063679063912262,
    "tp": 23790173,
    "fp": 11151125,
    "fn": 9889405
  },
  "flood_at_validation_threshold": {
    "threshold": 0.3,
    "iou": 0.5090347993855636,
    "f1": 0.6746495171520607,
    "precision": 0.6170184359993194,
    "recall": 0.7441555235638642,
    "tp": 25062844,
    "fp": 15556435,
    "fn": 8616734
  },
  "report_path": "C:\\NIT KKR\\after placement\\iit mandi hackathon\\outputs_e2e\\kuro_siwo_test_report.json"
}


## Interpretation checklist

- Compare the new run against the old run using the **same held-out test split** and report both argmax and validation-threshold metrics.
- Do not claim the high validation score generalizes to Nepal; the PDF shows four training shards and one validation shard, so validation variance can be high.
- Run ResNet18 as a controlled ablation only after this baseline is stable. Keep the same split, seed, preprocessing, scheduler, threshold policy, and stopping rule.
- Run class-weighted training as a **separate clean experiment** after inspecting `train_class_counts`. If `USE_CLASS_WEIGHTS` changes, reset the checkpoint.
- The full case-study workflow still needs the original source notebook if you want all Sentinel acquisition, HAND/slope filtering, infrastructure overlays, routing, dashboard, and EMSR927 cells preserved and patched.


In [1]:
![FloodRescue AI — Flood Detection Visualization](images/flood_prediction.png)

'[FloodRescue' is not recognized as an internal or external command,
operable program or batch file.


![FloodRescue AI — Flood Detection Visualization](images/flood_prediction.png)